# AfriVoices — TEST DE NORMALISATION (sans GPU, sur les paires déjà transcrites)

À exécuter **dans la même session que l'audit** (les paires `pairs[lang] = [(ref,pred),...]`
sont déjà en mémoire). On teste plusieurs normalisations appliquées **identiquement** à
ref ET pred, et on mesure le WER (et un CER) résultant par langue + macro.

⚠️ **Ce test est un DIAGNOSTIC, pas la soumission.** Il montre combien d'erreurs viennent
de la segmentation/orthographe plutôt que de l'écoute. On ne peut PAS normaliser la
référence à la soumission (Kaggle a sa propre référence) — mais **on peut** normaliser
notre PRED pour se rapprocher du style de référence. La dernière cellule teste ça
(normalisation de la pred SEULE), qui est la vraie question actionnable.

## 1. Normalisations candidates + mesure symétrique (ref+pred)

In [ ]:
import jiwer, re

def n_base(t):      # référence : nettoyage déjà fait, on ne touche pas
    return t.strip()

def n_nospace(t):   # supprime TOUTES les frontières de mots -> flux de caractères
    return re.sub(r"\s+","",t)

def n_som(t):       # somali : voyelles longues->courtes, dh->d, kh->k (variantes ortho)
    t=re.sub(r"([a-z])\1+", r"\1", t)   # double lettre -> simple (aa->a, oo->o)
    t=t.replace("dh","d").replace("kh","k").replace("q","k")
    return re.sub(r"\s+"," ",t).strip()

def n_som_nospace(t):
    return re.sub(r"\s+","", n_som(t))

def cer(refs, preds):
    # WER sur des caractères = CER
    r=[" ".join(list(x.replace(" ",""))) for x in refs]
    p=[" ".join(list(x.replace(" ",""))) for x in preds]
    return jiwer.wer(r,p)

NORMS = {
    "brut (WER actuel)":         n_base,
    "sans espaces (WER)":        n_nospace,
    "CER (caractères)":          None,          # traité à part
    "somali-ortho (WER)":        n_som,
    "somali-ortho sans espace":  n_som_nospace,
}

print(f"{'langue':6} " + " ".join(f"{k[:16]:>17}" for k in NORMS))
macro={k:[] for k in NORMS}
for lang, out in pairs.items():
    refs=[r for r,_ in out]; preds=[p for _,p in out]
    line=f"{lang:6} "
    for k, fn in NORMS.items():
        if k=="CER (caractères)":
            v=cer(refs,preds)
        else:
            v=jiwer.wer([fn(r) for r in refs],[fn(p) for p in preds])
        macro[k].append(v); line+=f"{v:17.3f} "
    print(line)
print("\nMACRO " + " ".join(f"{sum(macro[k])/len(macro[k]):17.3f}" for k in NORMS))
print("\nLecture:")
print("- 'sans espaces' << 'brut' -> une grosse part du WER = SEGMENTATION (frontières de mots)")
print("- 'CER' bas -> le modèle entend bien au niveau caractère (erreurs surtout ortho/espaces)")
print("- 'somali-ortho' << 'brut' pour som -> gain possible en normalisant les variantes")

## 2. La question actionnable : normaliser la PRED seule aide-t-elle ?

À la soumission, on ne contrôle que notre prédiction, pas la référence de Kaggle. Mais si
le style de référence est régulier (ex. voyelles longues systématiques en somali), aligner
notre pred sur ce style peut baisser le WER réel. On teste ici : ref BRUTE (comme Kaggle),
pred NORMALISÉE. Si le WER baisse, c'est un gain gratuit à la soumission.

In [ ]:
import jiwer, re

def collapse_spaces(t):   # normalise juste les espaces multiples (neutre)
    return re.sub(r"\s+"," ",t).strip()

# candidats de normalisation de la PRED SEULE (ref reste brute)
def p_dedup(t):           # dédouble les lettres (test : notre pred sur-double ?)
    return re.sub(r"([a-z\u0129\u0169])\1+", r"\1", collapse_spaces(t))

def p_som_long(t):        # somali : AJOUTE des voyelles longues (inverse) - test seulement
    return collapse_spaces(t)

TESTS = {
    "pred brute":          lambda t: collapse_spaces(t),
    "pred dédoublée":      p_dedup,
}

print("Effet d'une normalisation de la PRED SEULE (ref reste brute, comme à la soumission):\n")
print(f"{'langue':6} " + " ".join(f"{k:>16}" for k in TESTS))
macro={k:[] for k in TESTS}
for lang, out in pairs.items():
    refs=[r for r,_ in out]; preds=[p for _,p in out]
    line=f"{lang:6} "
    for k, fn in TESTS.items():
        v=jiwer.wer(refs,[fn(p) for p in preds])
        macro[k].append(v); line+=f"{v:16.3f} "
    print(line)
print("\nMACRO " + " ".join(f"{sum(macro[k])/len(macro[k]):16.3f}" for k in TESTS))
print("\nSi une colonne < 'pred brute' -> normalisation de pred = gain gratuit à la soumission.")
print("Sinon -> le gain 'segmentation' vu en §1 n'est PAS récupérable côté pred seule,")
print("         il faudrait que la RÉFÉRENCE Kaggle soit segmentée comme nous (non contrôlable).")

## 3. Interprétation

**Cas A — 'sans espaces' et 'CER' très bas (ex. moitié du WER brut)** : le modèle entend
bien, l'essentiel du WER est de la frontière de mots. Mais §2 dira si c'est récupérable :
- si normaliser la pred aide → on l'applique à la soumission (gain gratuit).
- si non → la référence Kaggle segmente autrement et on ne peut pas s'y aligner à l'aveugle.
  Piste : re-régler le **beta** du KenLM (favorise plus/moins d'espaces) et re-tester.

**Cas B — 'franche' confirmé (CER encore élevé)** : le modèle se trompe vraiment de sons.
Seul un meilleur backbone (omniASR) aide → décision architecture.

**Prochaine étape selon les chiffres** : soit on écrit la normalisation de soumission +
on re-règle beta et tu resoumets (rapide), soit on acte que le plafond est acoustique et
on pèse omniASR vs le temps restant.